# pricing-the-match, step 3: what does the operator's profit depend on?

**Question:** if we ran a book on a full season with our model and our prices, how much would we make, how bad could it get, and which levers matter most?

Step 2 ended on an uncomfortable number: at the bookmaker's own margin, a large share of our prices could be bet at a profit by someone better informed. This notebook puts money on that. It simulates a season of betting many times over and measures profit, risk, and how both respond to the things an operator controls.

| Step | Question |
|---|---|
| 1 | Can past results be turned into match probabilities? |
| 2 | How do probabilities become odds with a margin? |
| **3 (this notebook)** | What does the operator's profit depend on? |
| 4 | Web app and build log |

Self-contained: the first code cell repeats the step 1 and 2 functions.

In [ ]:
# --- Setup: step 1 and step 2 functions, unchanged (explained in those notebooks) ---
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import LogisticRegression

REQUIRED = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
ODDS = ["B365H", "B365D", "B365A"]


def load_matches(folder: str | Path) -> pd.DataFrame:
    """Read every CSV in `folder`, one file per season, into one table sorted by date."""
    files = sorted(Path(folder).glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files in {folder}")
    frames = []
    for f in files:
        df = pd.read_csv(f, encoding="latin-1")
        missing = [c for c in REQUIRED + ODDS if c not in df.columns]
        if missing:
            raise ValueError(f"{f.name} is missing columns {missing}")
        df = df[REQUIRED + ODDS].dropna(subset=REQUIRED)
        # football-data.co.uk mixes dd/mm/yy and dd/mm/yyyy across seasons
        df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
        df["season"] = f.stem
        frames.append(df)
    out = pd.concat(frames, ignore_index=True).sort_values("Date", kind="stable")
    out = out.dropna(subset=ODDS).reset_index(drop=True)
    bad = ~out["FTR"].isin(["H", "D", "A"])
    if bad.any():
        raise ValueError(f"{bad.sum()} rows have a result other than H/D/A")
    return out


def make_synthetic(folder: str | Path, seasons: int = 6, teams: int = 20,
                   margin: float = 0.05, seed: int = 7) -> None:
    """Write `seasons` fake league seasons in football-data.co.uk format.

    Each team has a hidden strength that drifts between seasons. Goals are
    Poisson. The fake bookmaker knows the true probabilities up to a little
    noise and adds `margin` on top - so on this data the bookmaker should
    beat any model, which is a useful sanity check.
    """
    rng = np.random.default_rng(seed)
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    names = [f"Team{i:02d}" for i in range(teams)]
    strength = rng.normal(0, 0.35, teams)
    goals = np.arange(0, 11)
    for s in range(seasons):
        rows = []
        day = pd.Timestamp(2019 + s, 8, 10)
        fixtures = [(h, a) for h in range(teams) for a in range(teams) if h != a]
        rng.shuffle(fixtures)
        for i, (h, a) in enumerate(fixtures):
            lam_h = np.exp(0.30 + strength[h] - strength[a])   # 0.30 = home edge
            lam_a = np.exp(0.05 + strength[a] - strength[h])
            ph = _pois(goals, lam_h)
            pa = _pois(goals, lam_a)
            grid = np.outer(ph, pa)
            p = np.array([np.tril(grid, -1).sum(), np.trace(grid), np.triu(grid, 1).sum()])
            p = p / p.sum()
            gh, ga = rng.poisson(lam_h), rng.poisson(lam_a)
            noisy = p * np.exp(rng.normal(0, 0.03, 3))
            noisy = noisy / noisy.sum()
            odds = np.round(1 / (noisy * (1 + margin)), 2)
            rows.append({
                "Date": (day + pd.Timedelta(days=i // 10 * 7)).strftime("%d/%m/%Y"),
                "HomeTeam": names[h], "AwayTeam": names[a],
                "FTHG": gh, "FTAG": ga,
                "FTR": "H" if gh > ga else ("D" if gh == ga else "A"),
                "B365H": odds[0], "B365D": odds[1], "B365A": odds[2],
            })
        pd.DataFrame(rows).to_csv(folder / f"synthetic_{2019 + s}.csv", index=False)
        strength = 0.85 * strength + rng.normal(0, 0.12, teams)


def _pois(k, lam):
    return poisson.pmf(k, lam)


START = 1500.0


def add_elo(matches: pd.DataFrame, k: float = 20.0, home_adv: float = 60.0) -> pd.DataFrame:
    """Return a copy of `matches` (must be date-sorted) with pre-match
    elo_home, elo_away and elo_diff (home minus away, before home advantage)."""
    if not matches["Date"].is_monotonic_increasing:
        raise ValueError("matches must be sorted by date")
    rating: dict[str, float] = {}
    eh, ea = [], []
    for home, away, res in zip(matches["HomeTeam"], matches["AwayTeam"], matches["FTR"]):
        rh, ra = rating.get(home, START), rating.get(away, START)
        eh.append(rh)
        ea.append(ra)
        expected_home = 1.0 / (1.0 + 10 ** (-(rh + home_adv - ra) / 400.0))
        score_home = {"H": 1.0, "D": 0.5, "A": 0.0}[res]
        delta = k * (score_home - expected_home)
        rating[home], rating[away] = rh + delta, ra - delta
    out = matches.copy()
    out["elo_home"], out["elo_away"] = eh, ea
    out["elo_diff"] = out["elo_home"] - out["elo_away"]
    return out


CLASSES = ["H", "D", "A"]
ODDS = ["B365H", "B365D", "B365A"]


def implied_probs(odds: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """odds (n,3) -> (fair probabilities (n,3), overround (n,))."""
    raw = 1.0 / np.asarray(odds, dtype=float)
    total = raw.sum(axis=1)
    return raw / total[:, None], total - 1.0


def split_by_season(df: pd.DataFrame, test_seasons: int = 1):
    """Train on the past, test on the most recent season(s). Never shuffle:
    a random split would let the model train on matches played after the
    ones it is tested on."""
    seasons = list(dict.fromkeys(df["season"]))
    if len(seasons) <= test_seasons:
        raise ValueError("need more seasons than test_seasons")
    test = df["season"].isin(seasons[-test_seasons:])
    return df[~test], df[test]


def fit(train: pd.DataFrame) -> LogisticRegression:
    model = LogisticRegression(C=1e6, max_iter=1000)  # C large = no regularisation
    model.fit(train[["elo_diff"]].to_numpy() / 100.0, train["FTR"])
    return model


def predict(model: LogisticRegression, df: pd.DataFrame) -> np.ndarray:
    """(n,3) probabilities in H, D, A order."""
    p = model.predict_proba(df[["elo_diff"]].to_numpy() / 100.0)
    order = [list(model.classes_).index(c) for c in CLASSES]
    return p[:, order]


def one_hot(results: pd.Series) -> np.ndarray:
    return np.stack([(results == c).to_numpy(dtype=float) for c in CLASSES], axis=1)


def brier(p: np.ndarray, y: np.ndarray) -> float:
    """Mean squared error of the probabilities. 0 = perfect, lower is better.
    Always saying 1/3 each scores 0.667."""
    return float(((p - y) ** 2).sum(axis=1).mean())


def log_loss(p: np.ndarray, y: np.ndarray) -> float:
    """Punishes confident wrong answers harder than Brier. Lower is better.
    Always saying 1/3 each scores 1.099."""
    return float(-(y * np.log(np.clip(p, 1e-12, 1))).sum(axis=1).mean())


def calibration(p: np.ndarray, y: np.ndarray, bins: int = 5) -> pd.DataFrame:
    """Pool all three outcomes, bucket by predicted probability, and compare
    with how often those outcomes happened. Calibrated = the columns match."""
    flat_p, flat_y = p.ravel(), y.ravel()
    edges = np.quantile(flat_p, np.linspace(0, 1, bins + 1))
    idx = np.clip(np.searchsorted(edges, flat_p, side="right") - 1, 0, bins - 1)
    t = pd.DataFrame({"bin": idx, "predicted": flat_p, "observed": flat_y})
    return t.groupby("bin").agg(predicted=("predicted", "mean"),
                                observed=("observed", "mean"),
                                n=("observed", "size")).round(3)


def evaluate(train: pd.DataFrame, test: pd.DataFrame, model: LogisticRegression) -> pd.DataFrame:
    """Score three forecasters on the test set: base rates, our model, the bookmaker."""
    y = one_hot(test["FTR"])
    base = np.tile(one_hot(train["FTR"]).mean(axis=0), (len(test), 1))
    ours = predict(model, test)
    book, _ = implied_probs(test[ODDS].to_numpy())
    rows = [(name, brier(p, y), log_loss(p, y))
            for name, p in [("base rates", base), ("elo + logistic", ours), ("bookmaker", book)]]
    return pd.DataFrame(rows, columns=["forecaster", "brier", "log_loss"]).round(4)


# ---- step 2: pricing ----
def _solve_power(p, target):
    """Find k per row so that sum_i p_i**k == target (bisection; the sum falls as k rises)."""
    p = np.atleast_2d(np.asarray(p, dtype=float))
    lo, hi = np.full(len(p), 0.01), np.full(len(p), 10.0)
    for _ in range(60):
        mid = (lo + hi) / 2
        too_big = (p ** mid[:, None]).sum(axis=1) > target
        lo, hi = np.where(too_big, mid, lo), np.where(too_big, hi, mid)
    return (lo + hi) / 2


def booked_probs(p, margin, method="proportional"):
    """Fair probabilities (n,3) -> inflated probabilities summing to 1 + margin."""
    p = np.atleast_2d(np.asarray(p, dtype=float))
    if margin < 0:
        raise ValueError("margin cannot be negative")
    if method == "proportional":
        return p * (1 + margin)
    if method == "power":
        return p ** _solve_power(p, 1 + margin)[:, None]
    raise ValueError(f"unknown method {method!r}")


def price(p, margin, method="proportional", tick=0.01):
    """Fair probabilities -> decimal odds offered to the customer.
    Odds are rounded DOWN to the tick, so rounding never gives margin away."""
    odds = 1.0 / booked_probs(p, margin, method)
    if tick:
        odds = np.floor(odds / tick + 1e-9) * tick
    return np.maximum(odds, 1.01)


def remove_margin(odds, method="proportional"):
    """The inverse: offered odds -> the fair probabilities behind them."""
    raw = 1.0 / np.atleast_2d(np.asarray(odds, dtype=float))
    if method == "proportional":
        return raw / raw.sum(axis=1, keepdims=True)
    if method == "power":
        return raw ** _solve_power(raw, 1.0)[:, None]
    raise ValueError(f"unknown method {method!r}")


def edge(odds, q):
    """Operator's expected profit per unit staked, if the true probability is q."""
    return 1.0 - np.asarray(q) * np.asarray(odds)

## 0. Data, model and the reference probabilities

**Where the data is looked for.** All paths are relative to the folder the notebook runs in: first `data/real`, then `../data/real` (the project's own data folder when the notebook is opened from `notebooks/`). If your files are somewhere else, add that folder to `CANDIDATES` in the next cell, also as a relative path. In Google Colab either upload the CSVs to a `data/real` folder in the session's file panel, or mount Google Drive (`from google.colab import drive; drive.mount("drive")`) and add `Path("drive/MyDrive/<your folder>")`.

In [ ]:
CANDIDATES = [Path("data/real"), Path("../data/real")]   # relative to where the notebook runs; add your own folder here
SYNTH = Path("data/synthetic")

DATA = next((c for c in CANDIDATES if list(c.glob("*.csv"))), None)
IS_REAL = DATA is not None
if IS_REAL:
    print(f"Using REAL season files from {DATA}")
else:
    DATA = SYNTH
    if not list(SYNTH.glob("*.csv")):
        make_synthetic(SYNTH)
    print("No real CSVs found, so using SYNTHETIC seasons (not real football)")

matches = add_elo(load_matches(DATA))
train, test = split_by_season(matches)
model = fit(train)

p_model = predict(model, test)                       # our probabilities, (n, 3) in H, D, A order
p_book, over_book = implied_probs(test[ODDS].to_numpy())
y_test = one_hot(test["FTR"])
print(f"train {len(train)} | test {len(test)} matches (season {test['season'].iloc[0]})")

# "Truth" for the simulation: the bookmaker's probabilities with the margin removed.
p_true = remove_margin(test[ODDS].to_numpy(), "proportional")

## 1. The simulation and its assumptions

Every number below depends on these assumptions, so they are stated up front. Each is a parameter you can change.

1. **Truth.** Match outcomes are drawn from the bookmaker's margin-free probabilities. The bookmaker is not the truth either, but it is the best forecaster we have, and what matters here is that somebody knows more than our model does.
2. **Casual bettors** bring a fixed turnover per match and split it across outcomes roughly in proportion to the odds on display (the public backs what the prices say is likely). They do not shop for value. `casual_noise` controls how unevenly the money lands: at 0 the book is perfectly balanced and the operator earns the same whoever wins, which never happens in practice. `casual_info` moves a share of the money to follow the true probabilities instead.
3. **Sharp bettors** know the true probabilities. They bet every price whose expected value beats `min_edge`, and they stake as much as we let them: `sharp_limit` per price.
4. **Prices are fixed.** We do not move odds when money arrives. Real operators do, so this is a worst case for the damage sharps can do.
5. **Price sensitivity.** Casual turnover shrinks as our margin rises above the market's: `turnover = base * exp(-elasticity * (margin - market margin))`. The elasticity is an **assumed** number, not estimated from data. Section 4 shows how much the answer depends on it.

Money is in arbitrary units. Read the results as ratios (hold %, loss probability), not as amounts.

In [ ]:
BASE = dict(
    margin=0.06,            # our overround
    method="power",         # how the margin is spread (step 2)
    casual_turnover=10_000, # casual money per match at the market margin
    casual_info=0.0,        # share of casual money that follows true probabilities
    casual_noise=0.3,       # how unevenly casual money lands (0 = perfectly balanced book)
    sharp_limit=200,        # max sharp stake we accept per price (0 = no sharps)
    min_edge=0.02,          # sharps only bet if expected value beats this
    elasticity=8.0,         # ASSUMED: how fast casual turnover falls as margin rises
    market_margin=0.055,    # the margin customers can get elsewhere
)


def build_book(p_op, p_true, margin, method, casual_turnover, casual_info, casual_noise,
               sharp_limit, min_edge, elasticity, market_margin, tick=0.01, seed=0):
    """Prices and stakes for every match. Returns (odds, casual stakes, sharp stakes), each (n, 3)."""
    odds = price(p_op, margin, method, tick=tick)
    shown = (1 / odds) / (1 / odds).sum(axis=1, keepdims=True)       # what the odds imply
    split = (1 - casual_info) * shown + casual_info * p_true
    if casual_noise:
        split = split * np.exp(np.random.default_rng(seed).normal(0, casual_noise, split.shape))
        split = split / split.sum(axis=1, keepdims=True)
    volume = casual_turnover * np.exp(-elasticity * (margin - market_margin))
    casual = volume * split
    sharp = np.where(p_true * odds - 1 > min_edge, float(sharp_limit), 0.0)
    return odds, casual, sharp


def expected_profit(odds, stakes, p_true):
    """Exact expected profit: each unit staked earns the operator 1 - (true prob * odds)."""
    return float((stakes * (1 - p_true * odds)).sum())


def simulate(p_op, p_true, n_sims=5000, seed=0, tick=0.01, **params):
    """Play the season n_sims times. Returns per-season operator profit and a summary."""
    cfg = {**BASE, **params}
    odds, casual, sharp = build_book(p_op, p_true, tick=tick, seed=seed, **cfg)
    stakes = casual + sharp
    rng = np.random.default_rng(seed + 1)
    cum = np.cumsum(p_true, axis=1)
    u = rng.random((n_sims, len(p_true)))
    winner = (u[:, :, None] > cum[None, :, :]).sum(axis=2).clip(max=2)   # (sims, matches)
    payout = np.take_along_axis((stakes * odds)[None, :, :], winner[:, :, None], axis=2)[:, :, 0]
    profit = stakes.sum() - payout.sum(axis=1)                         # per simulated season
    turnover = stakes.sum()
    return {
        "profit": profit,
        "turnover": turnover,
        "sharp turnover share": sharp.sum() / turnover,
        "expected profit": expected_profit(odds, stakes, p_true),
        "from casuals": expected_profit(odds, casual, p_true),
        "from sharps": expected_profit(odds, sharp, p_true),
        "hold %": 100 * profit.mean() / turnover,
        "worst 5% season": float(np.quantile(profit, 0.05)),
        "P(losing season)": float((profit < 0).mean()),
    }


def summary(label, r):
    return {"scenario": label, "turnover": round(r["turnover"]),
            "profit from casuals": round(r["from casuals"]), "profit from sharps": round(r["from sharps"]),
            "expected profit": round(r["expected profit"]), "hold %": round(r["hold %"], 2),
            "worst 5% season": round(r["worst 5% season"]), "P(losing season)": round(r["P(losing season)"], 3)}

## 2. Baseline: one season, played 5,000 times

Three books on the same fixtures, same margin:

- **casual only**: nobody is better informed than us
- **with sharps**: the baseline in `BASE`
- **with sharps, perfect model**: we price from the true probabilities, so there is nothing for sharps to find

In [ ]:
r_casual = simulate(p_model, p_true, sharp_limit=0)
r_base = simulate(p_model, p_true)
r_perfect = simulate(p_true, p_true)

print("REAL data" if IS_REAL else "SYNTHETIC data: shows the code works, not a finding")
print(f"{len(test)} matches | margin {BASE['margin']:.0%} ({BASE['method']}) | "
      f"sharp limit {BASE['sharp_limit']} per price vs casual turnover {BASE['casual_turnover']:,} per match")
print(f"Sharps bet {int((build_book(p_model, p_true, **BASE)[2] > 0).sum())} of {p_true.size} prices "
      f"and are {r_base['sharp turnover share']:.1%} of turnover\n")
baseline = pd.DataFrame([summary("casual only", r_casual), summary("with sharps", r_base),
                         summary("with sharps, perfect model", r_perfect)])
baseline

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
for r, label in [(r_casual, "casual only"), (r_base, "with sharps")]:
    ax.hist(r["profit"], bins=60, alpha=0.6, label=label)
ax.axvline(0, color="k", lw=1)
ax.set_xlabel("operator profit over the season"); ax.set_ylabel("simulated seasons")
ax.set_title("Same fixtures, same prices: season profit across 5,000 simulations")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

**What actually happened.** The simulation draws outcomes at random. The season was also really played, so here is the same book settled on the real results. This is a single draw from the distribution above, so treat it as a sanity check on the scale and not as a forecast.

In [ ]:
def settle(p_op, p_true, y, **params):
    """Profit of the book on the results that actually occurred (y is one-hot, (n, 3))."""
    odds, casual, sharp = build_book(p_op, p_true, **{**BASE, **params})
    stakes = casual + sharp
    return float(stakes.sum() - (stakes * odds * y).sum()), float(stakes.sum())

rows = []
for label, params in [("casual only", dict(sharp_limit=0)), ("with sharps", {})]:
    pnl, to = settle(p_model, p_true, y_test, **params)
    rows.append({"scenario": label, "turnover": round(to), "realised profit": round(pnl), "realised hold %": round(100 * pnl / to, 2)})
print(f"Settled on the actual results of season {test['season'].iloc[0]}:")
pd.DataFrame(rows)

## 3. Lever 1: how much sharp money to accept

The stake limit is the operator's most direct control. Everything else is held at the baseline.

In [ ]:
limits = [0, 50, 100, 200, 500, 1000, 2000, 5000]
lim = pd.DataFrame([{"sharp limit": L, **{k: v for k, v in summary("", simulate(p_model, p_true, sharp_limit=L)).items() if k != "scenario"}}
                    for L in limits])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lim["sharp limit"], lim["expected profit"], "o-", label="expected profit")
ax.plot(lim["sharp limit"], lim["worst 5% season"], "s--", label="worst 5% season")
ax.axhline(0, color="k", lw=1)
ax.set_xscale("symlog", linthresh=50)
ax.set_xlabel("sharp stake accepted per price"); ax.set_ylabel("season profit")
ax.set_title("Profit against the stake limit for sharp bettors")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

cost_per_unit = -simulate(p_model, p_true, n_sims=10, sharp_limit=1)["from sharps"]   # loss is linear in the limit
print(f"Each unit of sharp limit costs {cost_per_unit:,.1f} a season. Expected profit reaches zero at a limit of "
      f"about {r_casual['from casuals'] / cost_per_unit:,.0f} per price." if cost_per_unit > 0 else "Sharps find nothing to bet.")
lim

## 4. Lever 2: the margin

A higher margin earns more per bet and makes fewer prices exposed, but it drives casual customers away. That last effect is the assumed elasticity, so the plot shows three values of it. Where the peak sits depends heavily on that assumption; the shape of the trade-off does not.

In [ ]:
margins = np.arange(2, 26) / 100
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
best = []
for ax, L in zip(axes, [0, BASE["sharp_limit"]]):
    for el in [4.0, 8.0, 16.0]:
        prof = [simulate(p_model, p_true, n_sims=200, margin=m, sharp_limit=L, elasticity=el)["expected profit"] for m in margins]
        ax.plot(margins, prof, label=f"elasticity {el:g}")
        best.append({"sharp limit": L, "elasticity": el, "best margin": f"{margins[int(np.argmax(prof))]:.0%}",
                     "expected profit there": round(max(prof))})
    ax.axhline(0, color="k", lw=1)
    ax.axvline(BASE["market_margin"], color="k", ls=":", lw=1)
    ax.set_title("casual only" if L == 0 else f"with sharps (limit {L})")
    ax.set_xlabel("margin we charge"); ax.grid(alpha=0.3)
axes[0].set_ylabel("expected season profit"); axes[0].legend()
plt.show()
pd.DataFrame(best)

## 5. Lever 3: a better model

Suppose we could close part of the gap to the bookmaker. `quality` blends our probabilities with the true ones: 0 is the Elo model from step 1, 1 is a model as good as the reference. The log loss column shows what each step is worth in forecasting terms, and the profit column what it is worth in money.

In [ ]:
qual = []
for a in [0, 0.25, 0.5, 0.75, 1.0]:
    p_op = (1 - a) * p_model + a * p_true
    r = simulate(p_op, p_true)
    r0 = simulate(p_op, p_true, sharp_limit=5 * BASE["sharp_limit"])
    qual.append({"quality": a, "log loss": round(log_loss(p_op, y_test), 4),
                 "prices sharps bet": int((build_book(p_op, p_true, **BASE)[2] > 0).sum()),
                 "expected profit": round(r["expected profit"]), "hold %": round(r["hold %"], 2),
                 "P(losing season)": round(r["P(losing season)"], 3),
                 "expected profit, 5x sharp limit": round(r0["expected profit"])})
qual = pd.DataFrame(qual)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(qual["quality"], qual["expected profit"], "o-", label=f"sharp limit {BASE['sharp_limit']}")
ax.plot(qual["quality"], qual["expected profit, 5x sharp limit"], "s--", label=f"sharp limit {5 * BASE['sharp_limit']}")
ax.axhline(0, color="k", lw=1)
ax.set_xlabel("model quality (0 = our Elo model, 1 = as good as the reference)")
ax.set_ylabel("expected season profit"); ax.set_title("What a better model is worth")
ax.legend(); ax.grid(alpha=0.3)
plt.show()
qual

## 6. Which lever matters most?

One change at a time from the baseline, ranked by the effect on expected profit.

In [ ]:
half = 0.5 * p_model + 0.5 * p_true
scenarios = [
    ("baseline", p_model, {}),
    ("no sharps at all", p_model, dict(sharp_limit=0)),
    ("sharp limit x5", p_model, dict(sharp_limit=5 * BASE["sharp_limit"])),
    ("sharp limit halved", p_model, dict(sharp_limit=BASE["sharp_limit"] // 2)),
    ("margin 4%", p_model, dict(margin=0.04)),
    ("margin 10%", p_model, dict(margin=0.10)),
    ("proportional margin", p_model, dict(method="proportional")),
    ("model half-way to reference", half, {}),
    ("model as good as reference", p_true, {}),
    ("half of casual money is informed", p_model, dict(casual_info=0.5)),
]
sens = pd.DataFrame([summary(name, simulate(p, p_true, **kw)) for name, p, kw in scenarios])
sens["change vs baseline"] = sens["expected profit"] - sens.loc[0, "expected profit"]
print("REAL data" if IS_REAL else "SYNTHETIC data: shows the code works, not a finding")
sens.sort_values("change vs baseline", ascending=False).reset_index(drop=True)

## 7. Checks

The first check is the important one: the simulated average must agree with the exact expected value worked out by formula. If the two ever disagree, the simulation is wrong.

In [ ]:
rng = np.random.default_rng(1)
T = rng.dirichlet([4, 2.5, 3], size=200)                    # "true" probabilities
M = T * np.exp(rng.normal(0, 0.15, T.shape)); M /= M.sum(axis=1, keepdims=True)   # a noisy model of them

r = simulate(M, T, n_sims=20000, seed=3)
se = r["profit"].std() / np.sqrt(len(r["profit"]))
assert abs(r["profit"].mean() - r["expected profit"]) < 4 * se
print("ok  simulated mean profit matches the exact formula (within sampling error)")

r = simulate(M, T, n_sims=200, sharp_limit=0, casual_noise=0, method="proportional", margin=0.06, tick=None, elasticity=0)
assert np.isclose(r["expected profit"] / r["turnover"], 0.06 / 1.06)
assert r["profit"].std() < 1e-6 * r["turnover"]
print("ok  balanced book (casual money split like the odds): hold is exactly m / (1 + m) and the same whoever wins")

assert simulate(T, T, n_sims=10)["sharp turnover share"] == 0
print("ok  a perfect model gives sharps nothing to bet")

a, b = simulate(M, T, n_sims=50, sharp_limit=0), simulate(M, T, n_sims=50, min_edge=10.0)
assert np.allclose(a["profit"], b["profit"])
print("ok  a zero stake limit and an impossible edge threshold both switch sharps off")

prof = [simulate(M, T, n_sims=10, sharp_limit=L)["expected profit"] for L in (0, 100, 500, 2000)]
assert all(x > y for x, y in zip(prof, prof[1:]))
assert simulate(M, T, n_sims=10)["from sharps"] < 0
print("ok  every unit of sharp money costs the operator: profit falls as the limit rises")

assert np.allclose(simulate(M, T, n_sims=100, seed=5)["profit"], simulate(M, T, n_sims=100, seed=5)["profit"])
print("ok  same seed, same result")

y = np.eye(3)[rng.integers(0, 3, len(T))]
odds_, cas_, shp_ = build_book(M, T, **BASE)
by_hand = sum((cas_[i] + shp_[i]).sum() - ((cas_[i] + shp_[i]) * odds_[i] * y[i]).sum() for i in range(len(T)))
assert np.isclose(settle(M, T, y)[0], by_hand)
print("ok  settling on actual results matches a match-by-match calculation")

## 8. What this shows

Fill these in from your own run; the structure of the argument is:

- **Casual money pays the margin.** With no better-informed customers, the hold is close to $m/(1+m)$. The risk comes from the book being unbalanced, and it averages out over a season.
- **Sharp money is paid for by model error.** Every unit staked by someone who knows more costs us their edge. The damage scales with the stake limit and with how wrong the model is.
- **The three levers are substitutes.** A weak model can survive with low limits and a wide margin; a strong model can afford high limits and a thin margin, and takes more business as a result. That is why operators invest in pricing models.
- **What is not modelled:** moving prices as money arrives, in-play betting, bet types other than match result, customers who are neither casual nor sharp, and the true price sensitivity of casual customers.

**Step 4** wraps the three steps in a small web app with the levers as sliders, plus the build log.